In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# ============================================================
# 0. Paths
# ============================================================

sr386_path = Path("../datasets_metadata/csv/6.1_metadata_SR386_labels.csv")
sr1482_path = Path("../datasets_metadata/csv/6.2_metadata_SR1482_labels.csv")
histai_b1_path = Path("../datasets_metadata/csv/histai_colorectal_b1_metadata_regroup.csv")
histai_b2_path = Path("../datasets_metadata/csv/histai_colorectal_b2_metadata_regroup.csv")

output_dir = Path("output_crc_curated")
output_dir.mkdir(exist_ok=True)

# ============================================================
# 1. Load data
# ============================================================

sr386 = pd.read_csv(sr386_path)
sr1482 = pd.read_csv(sr1482_path)
histai_b1 = pd.read_csv(histai_b1_path)
histai_b2 = pd.read_csv(histai_b2_path)

# normalizar nombres de columnas
sr386.columns = [str(c).strip() for c in sr386.columns]
sr1482.columns = [str(c).strip() for c in sr1482.columns]
histai_b1.columns = [str(c).strip() for c in histai_b1.columns]
histai_b2.columns = [str(c).strip() for c in histai_b2.columns]

# ============================================================
# 2. Helpers generales
# ============================================================

def safe_col(df, col):
    return col in df.columns

def first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def norm_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def lower_text(x):
    return norm_text(x).lower()

def map_sex(x):
    t = lower_text(x)
    if t in ["m", "male"]:
        return "M"
    if t in ["f", "female"]:
        return "F"
    return pd.NA

def simplify_stage(x):
    t = lower_text(x)
    if t == "":
        return pd.NA
    if t.startswith("1"):
        return "I"
    if t.startswith("2"):
        return "II"
    if t.startswith("3"):
        return "III"
    if t.startswith("4"):
        return "IV"
    if t.startswith("a"):
        return pd.NA
    if "dukes" in t:
        if "a" in t:
            return "I"
        if "b" in t:
            return "II"
        if "c" in t:
            return "III"
        if "d" in t:
            return "IV"
    if t in ["a", "a1", "a2"]:
        return "I"
    if t in ["b", "b1", "b2"]:
        return "II"
    if t in ["c", "c1", "c2", "c3"]:
        return "III"
    if t in ["d"]:
        return "IV"
    return pd.NA

def map_site_group(x):
    t = lower_text(x)
    if t == "":
        return pd.NA

    if "rectosig" in t:
        return "rectosigmoid"
    if "rectum" in t or "rectal" in t or "lower rectum" in t or "upper rectum" in t:
        return "rectum"
    if "sigmoid" in t:
        return "sigmoid"
    if "descending" in t or "splenic flexure" in t or "left colon" in t:
        return "left_colon"
    if "transverse" in t:
        return "transverse_colon"
    if "ascending" in t or "caecum" in t or "cecum" in t or "hepatic flexure" in t or "right colon" in t:
        return "right_colon"
    if "colon" in t:
        return "colon_unspecified"
    if "appendix" in t:
        return "appendix"
    if "liver" in t or "hepatic" in t:
        return "metastatic_liver"
    if "peritone" in t or "omentum" in t:
        return "metastatic_peritoneal"
    if "lung" in t:
        return "metastatic_lung"
    return "other"

def map_side_from_site(site_group):
    if pd.isna(site_group):
        return pd.NA
    if site_group in ["right_colon", "caecum", "ascending"]:
        return "right"
    if site_group in ["left_colon", "sigmoid"]:
        return "left"
    if site_group == "transverse_colon":
        return "transverse"
    if site_group == "rectum":
        return "rectum"
    if site_group == "rectosigmoid":
        return "rectosigmoid"
    return pd.NA

def clean_grade(x):
    t = lower_text(x)
    if t == "":
        return pd.NA
    if "well" in t or "g1" in t:
        return "G1"
    if "mod" in t or "moder" in t or "g2" in t:
        return "G2"
    if "poor" in t or "low differentiation" in t or "g3" in t:
        return "G3"
    return pd.NA

def map_hist_type(x):
    t = lower_text(x)
    if t == "":
        return pd.NA
    if "signet" in t:
        return "signet_ring"
    if "mucin" in t:
        return "mucinous_adenocarcinoma"
    if "adenocarcinoma" in t or "a/c" in t:
        return "adenocarcinoma"
    if "adenoma" in t or "polyp" in t:
        return "adenoma_or_polyp"
    if "colitis" in t or "proctitis" in t:
        return "inflammatory"
    return "other"

def clean_pt(x):
    t = lower_text(x).replace(" ", "")
    if t == "":
        return pd.NA
    m = re.search(r'(yp)?pt([0-4][ab]?)', t)
    if m:
        return "pT" + m.group(2)
    m2 = re.search(r'\bt([0-4][ab]?)\b', t)
    if m2:
        return "T" + m2.group(1)
    return pd.NA

def clean_pn(x):
    t = lower_text(x).replace(" ", "")
    if t == "":
        return pd.NA
    m = re.search(r'(yp)?pn([0-2][abc]?)', t)
    if m:
        return "pN" + m.group(2)
    m2 = re.search(r'\bn([0-2][abc]?)\b', t)
    if m2:
        return "N" + m2.group(1)
    return pd.NA

def clean_pm(x):
    t = lower_text(x).replace(" ", "")
    if t == "":
        return pd.NA
    m = re.search(r'(yp)?pm([01][abcx]?)', t)
    if m:
        return "pM" + m.group(2)
    m2 = re.search(r'\bm([01][abcx]?)\b', t)
    if m2:
        return "M" + m2.group(1)
    if "cm1" in t:
        return "cM1"
    return pd.NA

def n_status_from_pn(x):
    t = lower_text(x).replace(" ", "")
    if t == "":
        return pd.NA
    if re.search(r'n0\b', t):
        return "N0"
    if re.search(r'n1', t):
        return "N+(1-3)"
    if re.search(r'n2', t):
        return "N+(>3)"
    return pd.NA

def mmr_status_from_text(x):
    t = lower_text(x)
    if t == "":
        return pd.NA
    if "loss" in t and "no loss" not in t:
        return "dMMR"
    if "no loss" in t:
        return "pMMR"
    if t in ["0", "0.0"]:
        return "pMMR"
    if t in ["1", "1.0"]:
        return "dMMR"
    return pd.NA

def msi_status_from_text(x):
    t = lower_text(x)
    if t == "":
        return pd.NA
    if "msi high" in t or "msi-h" in t:
        return "MSI-H"
    if "no msi" in t or "mss" in t:
        return "MSS_or_non_MSIH"
    if "not performed" in t:
        return pd.NA
    return pd.NA

def mutation_status(x):
    t = lower_text(x)
    if t == "":
        return pd.NA
    if "not performed" in t or "fail" in t:
        return pd.NA
    if "wt" in t or "no mutation" in t:
        return "WT"
    if "v600e" in t:
        return "V600E"
    if "mutation" in t or re.search(r'p\.\(', t) or re.search(r'c\.\d+', t) or "m (" in t:
        return "Mutated"
    return pd.NA

def lesion_class_from_taxonomy(x):
    t = lower_text(x)
    if t == "":
        return pd.NA
    if "malignant" in t:
        return "malignant"
    if "benign_precursor" in t:
        return "benign_precursor"
    if "inflammatory" in t or "non_neoplastic" in t:
        return "inflammatory_or_non_neoplastic"
    if "nonspecific_neoplastic" in t:
        return "other_neoplastic"
    return "other"

def bool_from_text_positive_negative(x, positive_terms, negative_terms):
    t = lower_text(x)
    if t == "":
        return pd.NA
    for neg in negative_terms:
        if neg in t:
            return 0
    for pos in positive_terms:
        if pos in t:
            return 1
    return pd.NA

# ============================================================
# 3. SR386: renombrar columnas esperadas si hiciera falta
# ============================================================

# Se intenta detectar nombres por posición si el csv vino sin header amigable.
# Si tu archivo ya tiene headers correctos, esto no altera nada relevante.

if sr386.shape[1] >= 30:
    expected_sr386 = [
        "case_id","death_5y","os_days","crc_primary_cause","age","sex",
        "site_of_tumour","site_of_tumour_grouping",
        "KRAS","NRAS","KRAS_exon2","KRAS_exon3","KRAS_exon4","BRAF",
        "MMR_IHC","MMR_loss_binary","primary_vs_met",
        "stage","stage_sub","pT","pN","pM",
        "tumour_type","differentiation",
        "peritoneal_involvement","distance_to_peritoneum",
        "margin_involved","vascular_lymphatic_invasion",
        "preop_rt","preop_ct"
    ]
    if len(sr386.columns) == len(expected_sr386):
        sr386.columns = expected_sr386

# ============================================================
# 4. Curación SR386
# ============================================================

sr386["dataset"] = "SR386"

case_col = first_existing_col(sr386, ["case_id", "Case ID", "case", "id"])
age_col = first_existing_col(sr386, ["age", "Age", "Age at Diagnosis"])
sex_col = first_existing_col(sr386, ["sex", "Sex", "gender"])
site_col = first_existing_col(sr386, ["site_of_tumour", "tumour_site", "site"])
site_group_col = first_existing_col(sr386, ["site_of_tumour_grouping", "site_group"])
grade_col = first_existing_col(sr386, ["differentiation", "grade"])
hist_col = first_existing_col(sr386, ["tumour_type", "histology"])
pt_col = first_existing_col(sr386, ["pT", "pt"])
pn_col = first_existing_col(sr386, ["pN", "pn"])
pm_col = first_existing_col(sr386, ["pM", "pm"])
stage_col = first_existing_col(sr386, ["stage", "Stage"])
mmr_col = first_existing_col(sr386, ["MMR_IHC", "MMR", "mmr_ihc"])
mmr_bin_col = first_existing_col(sr386, ["MMR_loss_binary", "mmr_loss_binary"])
kras_col = first_existing_col(sr386, ["KRAS", "kras"])
nras_col = first_existing_col(sr386, ["NRAS", "nras"])
braf_col = first_existing_col(sr386, ["BRAF", "braf"])
death5y_col = first_existing_col(sr386, ["death_5y", "death within 5 years", "dead_5y"])
os_col = first_existing_col(sr386, ["os_days", "days_until_death", "OS"])

sr386["case_id_norm"] = sr386[case_col] if case_col else sr386.index.astype(str)
sr386["age_norm"] = pd.to_numeric(sr386[age_col], errors="coerce") if age_col else np.nan
sr386["sex_norm"] = sr386[sex_col].apply(map_sex) if sex_col else pd.NA

sr386["site_group_norm"] = (
    sr386[site_group_col].apply(map_site_group) if site_group_col
    else sr386[site_col].apply(map_site_group) if site_col
    else pd.NA
)
sr386["side_norm"] = sr386["site_group_norm"].apply(map_side_from_site)

sr386["hist_type_norm"] = sr386[hist_col].apply(map_hist_type) if hist_col else pd.NA
sr386["grade_norm"] = sr386[grade_col].apply(clean_grade) if grade_col else pd.NA

sr386["pT_cat"] = sr386[pt_col].apply(clean_pt) if pt_col else pd.NA
sr386["pN_cat"] = sr386[pn_col].apply(clean_pn) if pn_col else pd.NA
sr386["pM_cat"] = sr386[pm_col].apply(clean_pm) if pm_col else pd.NA
sr386["N_status_norm"] = sr386["pN_cat"].apply(n_status_from_pn)

sr386["stage_simplified"] = sr386[stage_col].apply(simplify_stage) if stage_col else pd.NA

if mmr_bin_col:
    sr386["MMR_status_norm"] = sr386[mmr_bin_col].apply(
        lambda x: "dMMR" if str(x).strip() in ["1", "1.0"] else "pMMR" if str(x).strip() in ["0", "0.0"] else pd.NA
    )
elif mmr_col:
    sr386["MMR_status_norm"] = sr386[mmr_col].apply(mmr_status_from_text)
else:
    sr386["MMR_status_norm"] = pd.NA

sr386["MSI_status_norm"] = pd.NA  # no siempre está explícito en SR386
sr386["KRAS_status_norm"] = sr386[kras_col].apply(mutation_status) if kras_col else pd.NA
sr386["NRAS_status_norm"] = sr386[nras_col].apply(mutation_status) if nras_col else pd.NA
sr386["BRAF_status_norm"] = sr386[braf_col].apply(mutation_status) if braf_col else pd.NA

sr386["death_5y_norm"] = pd.to_numeric(sr386[death5y_col], errors="coerce") if death5y_col else np.nan
sr386["os_days_norm"] = pd.to_numeric(sr386[os_col], errors="coerce") if os_col else np.nan

# ============================================================
# 5. Curación SR1482
# ============================================================

sr1482["dataset"] = "SR1482"

if sr1482.shape[1] == 13:
    sr1482.columns = [
        "case_id","tumour_site","stage_notes","dukes","pT","pN","pM",
        "sex","age","MMR","MSI","KRAS","NRAS","BRAF"
    ][:sr1482.shape[1]]

# si vino con 14 columnas reales, rehacemos
if sr1482.shape[1] == 14:
    sr1482.columns = [
        "case_id","tumour_site","stage_notes","dukes","pT","pN","pM",
        "sex","age","MMR","MSI","KRAS","NRAS","BRAF"
    ]

case_col = first_existing_col(sr1482, ["case_id", "Case ID", "case", "id"])
age_col = first_existing_col(sr1482, ["age", "Age"])
sex_col = first_existing_col(sr1482, ["sex", "Sex"])
site_col = first_existing_col(sr1482, ["tumour_site", "site"])
dukes_col = first_existing_col(sr1482, ["dukes", "Dukes"])
pt_col = first_existing_col(sr1482, ["pT"])
pn_col = first_existing_col(sr1482, ["pN"])
pm_col = first_existing_col(sr1482, ["pM"])
mmr_col = first_existing_col(sr1482, ["MMR"])
msi_col = first_existing_col(sr1482, ["MSI"])
kras_col = first_existing_col(sr1482, ["KRAS"])
nras_col = first_existing_col(sr1482, ["NRAS"])
braf_col = first_existing_col(sr1482, ["BRAF"])
notes_col = first_existing_col(sr1482, ["stage_notes", "notes"])

sr1482["case_id_norm"] = sr1482[case_col] if case_col else sr1482.index.astype(str)
sr1482["age_norm"] = pd.to_numeric(sr1482[age_col], errors="coerce") if age_col else np.nan
sr1482["sex_norm"] = sr1482[sex_col].apply(map_sex) if sex_col else pd.NA

sr1482["site_group_norm"] = sr1482[site_col].apply(map_site_group) if site_col else pd.NA
sr1482["side_norm"] = sr1482["site_group_norm"].apply(map_side_from_site)

sr1482["pT_cat"] = sr1482[pt_col].apply(clean_pt) if pt_col else pd.NA
sr1482["pN_cat"] = sr1482[pn_col].apply(clean_pn) if pn_col else pd.NA
sr1482["pM_cat"] = sr1482[pm_col].apply(clean_pm) if pm_col else pd.NA
sr1482["N_status_norm"] = sr1482["pN_cat"].apply(n_status_from_pn)

sr1482["stage_simplified"] = sr1482[dukes_col].apply(simplify_stage) if dukes_col else pd.NA
sr1482["MMR_status_norm"] = sr1482[mmr_col].apply(mmr_status_from_text) if mmr_col else pd.NA
sr1482["MSI_status_norm"] = sr1482[msi_col].apply(msi_status_from_text) if msi_col else pd.NA
sr1482["KRAS_status_norm"] = sr1482[kras_col].apply(mutation_status) if kras_col else pd.NA
sr1482["NRAS_status_norm"] = sr1482[nras_col].apply(mutation_status) if nras_col else pd.NA
sr1482["BRAF_status_norm"] = sr1482[braf_col].apply(mutation_status) if braf_col else pd.NA

sr1482["hist_type_norm"] = sr1482[notes_col].apply(map_hist_type) if notes_col else pd.NA
sr1482["grade_norm"] = sr1482[notes_col].apply(clean_grade) if notes_col else pd.NA
sr1482["lesion_class_norm"] = "malignant"

# ============================================================
# 6. HISTAI helpers
# ============================================================

def make_full_text(df):
    text_cols = [c for c in ["diagnosis", "conclusion", "grossing", "micro_protocol", "additional_info"] if c in df.columns]
    def join_row(row):
        vals = []
        for c in text_cols:
            if pd.notna(row[c]):
                vals.append(str(row[c]))
        return " ".join(vals)
    return df.apply(join_row, axis=1)

def extract_grade_from_text(text):
    t = lower_text(text)
    if re.search(r'\bg1\b', t) or "grade: 1" in t or "1-high grade differentiation" in t:
        return "G1"
    if re.search(r'\bg2\b', t) or "grade: 2" in t:
        return "G2"
    if re.search(r'\bg3\b', t) or "grade: 3" in t or "low differentiation" in t:
        return "G3"
    if "well differentiated" in t:
        return "G1"
    if "moderately differentiated" in t:
        return "G2"
    if "poorly differentiated" in t:
        return "G3"
    return pd.NA

def extract_hist_type_from_text(text):
    t = lower_text(text)
    if "signet" in t:
        return "signet_ring"
    if "mucinous" in t:
        return "mucinous_adenocarcinoma"
    if "adenocarcinoma" in t:
        return "adenocarcinoma"
    if "adenoma" in t or "polyp" in t or "serrated" in t:
        return "adenoma_or_polyp"
    if "colitis" in t or "proctitis" in t:
        return "inflammatory"
    return pd.NA

def extract_pt_from_text(text):
    t = lower_text(text).replace(" ", "")
    m = re.search(r'pt([0-4][ab]?)', t)
    if m:
        return "pT" + m.group(1)
    m2 = re.search(r'\bt([0-4][ab]?)\b', t)
    if m2:
        return "T" + m2.group(1)
    return pd.NA

def extract_pn_from_text(text):
    t = lower_text(text).replace(" ", "")
    m = re.search(r'pn([0-2][abc]?)', t)
    if m:
        return "pN" + m.group(1)
    m2 = re.search(r'\bn([0-2][abc]?)\b', t)
    if m2:
        return "N" + m2.group(1)
    return pd.NA

def extract_pm_from_text(text):
    t = lower_text(text).replace(" ", "")
    m = re.search(r'pm([01][abcx]?)', t)
    if m:
        return "pM" + m.group(1)
    m2 = re.search(r'\bm([01][abcx]?)\b', t)
    if m2:
        return "M" + m2.group(1)
    return pd.NA

def extract_ln_examined(text):
    t = lower_text(text)
    patterns = [
        r'total number:\s*(\d+)',
        r'number examined:\s*(\d+)',
        r'(\d+)\s*lymph nodes examined',
    ]
    for p in patterns:
        m = re.search(p, t)
        if m:
            return float(m.group(1))
    m2 = re.search(r'n\d+[ab]?\((\d+)/(\d+)\)', t.replace(" ", ""))
    if m2:
        return float(m2.group(2))
    return np.nan

def extract_ln_positive(text):
    t = lower_text(text)
    patterns = [
        r'number of involved lymph nodes:\s*(\d+)',
        r'number of affected lymph nodes:\s*(\d+)',
        r'number with metastases:\s*(\d+)',
    ]
    for p in patterns:
        m = re.search(p, t)
        if m:
            return float(m.group(1))
    m2 = re.search(r'n\d+[ab]?\((\d+)/(\d+)\)', t.replace(" ", ""))
    if m2:
        return float(m2.group(1))
    return np.nan

def n_status_from_count(pos):
    if pd.isna(pos):
        return pd.NA
    if pos == 0:
        return "N0"
    if pos <= 3:
        return "N+(1-3)"
    return "N+(>3)"

def extract_lvi(text):
    t = lower_text(text)
    if "lymphovascular invasion" in t or "venous invasion" in t or "lvi1" in t or "lv1" in t:
        if "not identified" in t and ("lymphovascular invasion" in t or "venous invasion" in t):
            return 0
        return 1
    if "lvi0" in t or "lv0" in t:
        return 0
    return pd.NA

def extract_pni(text):
    t = lower_text(text)
    if "perineural invasion: present" in t or "pni1" in t or "pn1" in t:
        return 1
    if "perineural invasion: not identified" in t or "pni0" in t or "pn0" in t:
        return 0
    return pd.NA

def histai_curate(df, dataset_name):
    df = df.copy()
    df["dataset"] = dataset_name

    case_col = first_existing_col(df, ["case_id", "id"])
    age_col = first_existing_col(df, ["age", "Age"])
    sex_col = first_existing_col(df, ["gender", "sex", "Sex"])
    dx_norm_col = first_existing_col(df, ["diagnosis_normalized"])
    tax_group_col = first_existing_col(df, ["taxonomy_group"])
    tax_subgroup_col = first_existing_col(df, ["taxonomy_subgroup"])
    tax_site_col = first_existing_col(df, ["taxonomy_site"])

    df["case_id_norm"] = df[case_col] if case_col else df.index.astype(str)
    df["age_norm"] = pd.to_numeric(df[age_col], errors="coerce") if age_col else np.nan
    df["sex_norm"] = df[sex_col].apply(map_sex) if sex_col else pd.NA

    df["full_text"] = make_full_text(df)

    df["site_group_norm"] = df[tax_site_col].apply(map_site_group) if tax_site_col else df["full_text"].apply(map_site_group)
    df["side_norm"] = df["site_group_norm"].apply(map_side_from_site)

    df["lesion_class_norm"] = df[tax_group_col].apply(lesion_class_from_taxonomy) if tax_group_col else pd.NA

    if tax_subgroup_col:
        df["hist_type_norm"] = df[tax_subgroup_col].apply(map_hist_type)
        df["hist_type_norm"] = df["hist_type_norm"].fillna(df["full_text"].apply(extract_hist_type_from_text))
    else:
        df["hist_type_norm"] = df["full_text"].apply(extract_hist_type_from_text)

    df["grade_norm"] = df["full_text"].apply(extract_grade_from_text)

    df["pT_cat"] = df["full_text"].apply(extract_pt_from_text)
    df["pN_cat"] = df["full_text"].apply(extract_pn_from_text)
    df["pM_cat"] = df["full_text"].apply(extract_pm_from_text)

    df["LN_examined_norm"] = df["full_text"].apply(extract_ln_examined)
    df["LN_positive_norm"] = df["full_text"].apply(extract_ln_positive)

    df["N_status_norm"] = df["LN_positive_norm"].apply(n_status_from_count)

    # fallback con pN si no hay conteos
    mask_missing_n = df["N_status_norm"].isna()
    df.loc[mask_missing_n, "N_status_norm"] = df.loc[mask_missing_n, "pN_cat"].apply(n_status_from_pn)

    # stage simplificado
    def derive_stage(row):
        pm = lower_text(row["pM_cat"])
        pn = lower_text(row["pN_cat"])
        pt = lower_text(row["pT_cat"])
        if "m1" in pm:
            return "IV"
        if "n1" in pn or "n2" in pn:
            return "III"
        if "t3" in pt or "t4" in pt:
            return "II"
        if "t1" in pt or "t2" in pt:
            return "I"
        return pd.NA

    df["stage_simplified"] = df.apply(derive_stage, axis=1)

    df["LVI_norm"] = df["full_text"].apply(extract_lvi)
    df["PNI_norm"] = df["full_text"].apply(extract_pni)

    df["MMR_status_norm"] = pd.NA
    df["MSI_status_norm"] = pd.NA
    df["KRAS_status_norm"] = pd.NA
    df["NRAS_status_norm"] = pd.NA
    df["BRAF_status_norm"] = pd.NA
    df["death_5y_norm"] = np.nan
    df["os_days_norm"] = np.nan

    return df

# ============================================================
# 7. Curación HISTAI b1 y b2
# ============================================================

histai_b1_norm = histai_curate(histai_b1, "HISTAI_CRC_B1")
histai_b2_norm = histai_curate(histai_b2, "HISTAI_CRC_B2")

# ============================================================
# 8. Common schema
# ============================================================

common_cols = [
    "case_id_norm", "dataset",
    "age_norm", "sex_norm",
    "site_group_norm", "side_norm",
    "lesion_class_norm",
    "hist_type_norm", "grade_norm",
    "pT_cat", "pN_cat", "pM_cat",
    "N_status_norm", "stage_simplified",
    "LN_examined_norm", "LN_positive_norm",
    "LVI_norm", "PNI_norm",
    "MMR_status_norm", "MSI_status_norm",
    "KRAS_status_norm", "NRAS_status_norm", "BRAF_status_norm",
    "death_5y_norm", "os_days_norm"
]

def ensure_common_cols(df, cols):
    for c in cols:
        if c not in df.columns:
            df[c] = pd.NA
    return df[cols].copy()

sr386_common = ensure_common_cols(sr386, common_cols)
sr1482_common = ensure_common_cols(sr1482, common_cols)
histai_b1_common = ensure_common_cols(histai_b1_norm, common_cols)
histai_b2_common = ensure_common_cols(histai_b2_norm, common_cols)

crc_all_common = pd.concat(
    [sr386_common, sr1482_common, histai_b1_common, histai_b2_common],
    ignore_index=True
)

# ============================================================
# 9. Guardar copias nuevas
# ============================================================

sr386.to_csv(output_dir / "6.1_metadata_SR386_labels_norm.csv", index=False)
sr1482.to_csv(output_dir / "6.2_metadata_SR1482_labels_norm.csv", index=False)
histai_b1_norm.to_csv(output_dir / "histai_colorectal_b1_metadata_regroup_norm.csv", index=False)
histai_b2_norm.to_csv(output_dir / "histai_colorectal_b2_metadata_regroup_norm.csv", index=False)

sr386_common.to_csv(output_dir / "SR386_common.csv", index=False)
sr1482_common.to_csv(output_dir / "SR1482_common.csv", index=False)
histai_b1_common.to_csv(output_dir / "HISTAI_CRC_B1_common.csv", index=False)
histai_b2_common.to_csv(output_dir / "HISTAI_CRC_B2_common.csv", index=False)

crc_all_common.to_csv(output_dir / "crc_all_common.csv", index=False)

print("Curación completada.")
print(f"Archivos guardados en: {output_dir.resolve()}")
print("Tamaños:")
print("SR386_common:", sr386_common.shape)
print("SR1482_common:", sr1482_common.shape)
print("HISTAI_CRC_B1_common:", histai_b1_common.shape)
print("HISTAI_CRC_B2_common:", histai_b2_common.shape)
print("crc_all_common:", crc_all_common.shape)

Curación completada.
Archivos guardados en: C:\Users\vicen\Repositories\Tesis\Scripts\output_crc_curated
Tamaños:
SR386_common: (427, 25)
SR1482_common: (416, 25)
HISTAI_CRC_B1_common: (878, 25)
HISTAI_CRC_B2_common: (57, 25)
crc_all_common: (1778, 25)


C:\Users\vicen\AppData\Local\Temp\ipykernel_27868\1496076772.py:669: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  crc_all_common = pd.concat(
